# MVP-2: Exploración de matrices ómicas

Verificar dimensiones, formatos, cadenas de join y estadísticas de cada archivo
descargado de GEO antes de construir el manifest MVP-2.

**Ejecutar celda por celda.**

## 0. CONFIG

In [2]:
from pathlib import Path

CONFIG = {
    'csv_central': Path('/Users/JCB/Documentos/Proyecto Integrador/data/Lifespan_Study_Data.csv'),
    'geo_rna_meta': Path('/Users/JCB/Documentos/Proyecto Integrador/data/GSE179848_sample_metadata.csv'),
    'geo_met_meta': Path('/Users/JCB/Documentos/Proyecto Integrador/data/GSE179847_sample_metadata.csv'),
    'rna_processed': Path('/Volumes/SanDisk SSD 1TB/Storage/Data/GSE179848/GSE179848_processed_cell_lifespan_RNAseq_data.csv.gz'),
    'rna_raw_counts': Path('/Volumes/SanDisk SSD 1TB/Storage/Data/GSE179848/GSE179848_raw_counts_cell_lifespan_RNAseq_data.csv.gz'),
    'met_processed': Path('/Volumes/SanDisk SSD 1TB/Storage/Data/GSE179847/GSE179847_Cell_lifespan_DNAm_processed_matrix.csv.gz'),
    'manifest_mvp1': Path('/Users/JCB/Documentos/Proyecto Integrador/data/manifests/manifest_full_20260327_152829.csv'),
}

# Verificar que existen
for name, path in CONFIG.items():
    exists = "✅" if path.exists() else "❌ NO ENCONTRADO"
    print(f"  {exists}  {name}: {path.name}")

  ✅  csv_central: Lifespan_Study_Data.csv
  ✅  geo_rna_meta: GSE179848_sample_metadata.csv
  ✅  geo_met_meta: GSE179847_sample_metadata.csv
  ✅  rna_processed: GSE179848_processed_cell_lifespan_RNAseq_data.csv.gz
  ✅  rna_raw_counts: GSE179848_raw_counts_cell_lifespan_RNAseq_data.csv.gz
  ✅  met_processed: GSE179847_Cell_lifespan_DNAm_processed_matrix.csv.gz
  ✅  manifest_mvp1: manifest_full_20260327_152829.csv


## 1. Imports

In [3]:
import pandas as pd
import numpy as np
import gzip

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

## 2. Inspeccionar headers de matrices (sin cargar completas)

Lectura ligera — solo primeras líneas de cada archivo.

In [4]:
print("=" * 70)
print("INSPECCIÓN DE HEADERS")
print("=" * 70)

# --- RNA processed ---
print("\n--- RNA PROCESSED ---")
with gzip.open(CONFIG['rna_processed'], 'rt') as f:
    rna_header_line = f.readline().strip()
    rna_row1 = f.readline().strip()
    rna_row2 = f.readline().strip()

rna_cols_raw = rna_header_line.split(',')
print(f"  N columnas (incl. index): {len(rna_cols_raw)}")
print(f"  Col 0 (index): {rna_cols_raw[0]}")
print(f"  Col 1-5: {rna_cols_raw[1:6]}")
print(f"  Última col: {rna_cols_raw[-1]}")

# Extraer los números de Sample_N
sample_nums_rna = []
for c in rna_cols_raw[1:]:
    c_clean = c.strip().strip('"')
    if c_clean.startswith('Sample_'):
        try:
            sample_nums_rna.append(int(c_clean.replace('Sample_', '')))
        except:
            pass
print(f"  Sample nums rango: {min(sample_nums_rna)}-{max(sample_nums_rna)} ({len(sample_nums_rna)} muestras)")

# Primera fila de datos
row1_parts = rna_row1.split(',')
print(f"  Gen 1 (index): {row1_parts[0]}")
print(f"  Valores 1-3: {row1_parts[1:4]}")

# --- RNA raw counts ---
print("\n--- RNA RAW COUNTS ---")
with gzip.open(CONFIG['rna_raw_counts'], 'rt') as f:
    raw_header = f.readline().strip()
    raw_row1 = f.readline().strip()

raw_cols = raw_header.split(',')
print(f"  N columnas: {len(raw_cols)}")
print(f"  Col 0: {raw_cols[0]}")
print(f"  Col 1-3: {raw_cols[1:4]}")
raw_parts = raw_row1.split(',')
print(f"  Gen 1: {raw_parts[0]}")
print(f"  Valores 1-3: {raw_parts[1:4]}")

# --- METILACIÓN (solo header) ---
print("\n--- METILACIÓN PROCESSED ---")
with gzip.open(CONFIG['met_processed'], 'rt') as f:
    met_header_line = f.readline().strip()
    met_row1_partial = f.readline()[:500]

met_cols_raw = met_header_line.split(',')
print(f"  N columnas (incl. index): {len(met_cols_raw)}")
print(f"  Col 0: {met_cols_raw[0]}")
print(f"  Col 1: {met_cols_raw[1]}")
print(f"  Col 2: {met_cols_raw[2]}")
print(f"  Última col: {met_cols_raw[-1]}")

# Separar beta de pval
beta_cols_all = [c.strip().strip('"') for c in met_cols_raw[1:] if 'beta' in c.lower()]
pval_cols_all = [c.strip().strip('"') for c in met_cols_raw[1:] if 'pval' in c.lower() or 'detection' in c.lower()]
print(f"\n  Columnas tipo 'beta': {len(beta_cols_all)}")
print(f"  Columnas tipo 'Detection Pval': {len(pval_cols_all)}")
print(f"  Beta ejemplo: {beta_cols_all[0] if beta_cols_all else 'N/A'}")
print(f"  Pval ejemplo: {pval_cols_all[0] if pval_cols_all else 'N/A'}")

# Extraer sentrix basenames
sentrix_from_matrix = []
for c in beta_cols_all:
    parts = c.rsplit(' ', 1)
    if len(parts) == 2:
        sentrix_from_matrix.append(parts[0])
print(f"\n  Sentrix basenames únicos en matriz: {len(set(sentrix_from_matrix))}")
print(f"  Ejemplos: {sentrix_from_matrix[:5]}")

# Primera fila de datos
met_parts = met_row1_partial.split(',')
print(f"\n  CpG 1 (index): {met_parts[0]}")
print(f"  Valor 1-2: {met_parts[1:3]}")

INSPECCIÓN DE HEADERS

--- RNA PROCESSED ---
  N columnas (incl. index): 346
  Col 0 (index): ""
  Col 1-5: ['"Sample_1"', '"Sample_2"', '"Sample_3"', '"Sample_4"', '"Sample_5"']
  Última col: "Sample_360"
  Sample nums rango: 1-360 (345 muestras)
  Gen 1 (index): "A1BG"
  Valores 1-3: ['8.06052905511525', '7.67918770316003', '8.24693240284204']

--- RNA RAW COUNTS ---
  N columnas: 346
  Col 0: ""
  Col 1-3: ['"Sample_1"', '"Sample_2"', '"Sample_3"']
  Gen 1: "NR_046018"
  Valores 1-3: ['5.27129', '8.08786', '0.196922']

--- METILACIÓN PROCESSED ---
  N columnas (incl. index): 959
  Col 0: ""
  Col 1: "203784950092_R01C01 beta"
  Col 2: "203784950092_R01C01 Detection Pval"
  Última col: "203833040058_R07C01 Detection Pval"

  Columnas tipo 'beta': 479
  Columnas tipo 'Detection Pval': 479
  Beta ejemplo: 203784950092_R01C01 beta
  Pval ejemplo: 203784950092_R01C01 Detection Pval

  Sentrix basenames únicos en matriz: 479
  Ejemplos: ['203784950092_R01C01', '203784950092_R02C01', '2037

## 3. Cargar RNA-seq processed (~58 MB, rápido)

In [5]:
rna = pd.read_csv(CONFIG['rna_processed'], compression='gzip', index_col=0)
print(f"  Shape: {rna.shape}")
print(f"  Orientación: {'genes × muestras' if rna.shape[0] > rna.shape[1] else 'muestras × genes'}")
print(f"  Index dtype: {rna.index.dtype}, name: {rna.index.name}")
print(f"  Primeros 5 genes: {rna.index[:5].tolist()}")
print(f"  Últimos 5 genes: {rna.index[-5:].tolist()}")
print(f"  Primeras 5 columnas: {rna.columns[:5].tolist()}")
print(f"  Últimas 5 columnas: {rna.columns[-5:].tolist()}")

# Estadísticas
print(f"\n  Rango global: [{rna.values.min():.4f}, {rna.values.max():.4f}]")
print(f"  Media global: {rna.values.mean():.4f}")
print(f"  NaN total: {rna.isna().sum().sum()}")
print(f"  % NaN: {rna.isna().sum().sum() / rna.size * 100:.4f}%")

# ¿Son valores VST? VST produce valores en rango ~4-16 (log2-like)
vals_flat = rna.values.flatten()
vals_flat = vals_flat[~np.isnan(vals_flat)]
print(f"\n  Percentiles:")
for p in [1, 5, 25, 50, 75, 95, 99]:
    print(f"    P{p}: {np.percentile(vals_flat, p):.4f}")

if 4 < np.median(vals_flat) < 20:
    print("\n  ✅ Valores parecen VST-normalizados (rango log2-like)")
elif np.median(vals_flat) > 100:
    print("\n  ⚠️ Valores parecen counts crudos o FPKM (mediana > 100)")
else:
    print(f"\n  ⚠️ Rango no estándar — verificar manualmente")

# Varianza por gen
gene_var = rna.var(axis=1).sort_values(ascending=False)
print(f"\n  Genes con varianza > 0: {(gene_var > 0).sum()} / {len(gene_var)}")
print(f"  Top 10 genes más variables:")
for g, v in gene_var.head(10).items():
    print(f"    {g}: var={v:.4f}")

  Shape: (28633, 345)
  Orientación: genes × muestras
  Index dtype: str, name: None
  Primeros 5 genes: ['A1BG', 'A1BG-AS1', 'A1CF', 'A2M', 'A2M-AS1']
  Últimos 5 genes: ['ZYG11A', 'ZYG11B', 'ZYX', 'ZZEF1', 'ZZZ3']
  Primeras 5 columnas: ['Sample_1', 'Sample_2', 'Sample_3', 'Sample_4', 'Sample_5']
  Últimas 5 columnas: ['Sample_356', 'Sample_357', 'Sample_358', 'Sample_359', 'Sample_360']

  Rango global: [3.7230, 22.3335]
  Media global: 6.5740
  NaN total: 0
  % NaN: 0.0000%

  Percentiles:
    P1: 3.7230
    P5: 3.7230
    P25: 4.1274
    P50: 5.8972
    P75: 8.7000
    P95: 11.2803
    P99: 13.0668

  ✅ Valores parecen VST-normalizados (rango log2-like)

  Genes con varianza > 0: 28633 / 28633
  Top 10 genes más variables:
    XIST: var=17.8838
    CADM3: var=13.6403
    RPS4Y1: var=9.8690
    ZBTB16: var=9.8484
    RNA28SN5: var=9.8375
    IGF2: var=9.7971
    USP9Y: var=9.1743
    CORIN: var=8.8031
    CNR1: var=8.3681
    NLGN4Y: var=7.5642


## 4. Cargar RNA raw counts (para comparar)

In [6]:
rna_raw = pd.read_csv(CONFIG['rna_raw_counts'], compression='gzip', index_col=0)
print(f"  Shape: {rna_raw.shape}")
print(f"  Primeros 5 genes: {rna_raw.index[:5].tolist()}")
print(f"  Primeras 5 cols: {rna_raw.columns[:5].tolist()}")
print(f"  Rango: [{rna_raw.values.min()}, {rna_raw.values.max()}]")
print(f"  NaN: {rna_raw.isna().sum().sum()}")

# ¿Mismos genes y muestras que processed?
genes_match = set(rna.index) == set(rna_raw.index)
cols_match = set(rna.columns) == set(rna_raw.columns)
print(f"\n  Mismos genes que processed: {genes_match}")
print(f"  Mismas columnas que processed: {cols_match}")
if not genes_match:
    only_processed = set(rna.index) - set(rna_raw.index)
    only_raw = set(rna_raw.index) - set(rna.index)
    print(f"    Solo en processed: {len(only_processed)} genes")
    print(f"    Solo en raw: {len(only_raw)} genes")

  Shape: (70551, 345)
  Primeros 5 genes: ['NR_046018', 'NR_024540', 'NR_106918', 'NR_036051', 'NR_026818']
  Primeras 5 cols: ['Sample_1', 'Sample_2', 'Sample_3', 'Sample_4', 'Sample_5']
  Rango: [0.0, 3086320.0]
  NaN: 0

  Mismos genes que processed: False
  Mismas columnas que processed: True
    Solo en processed: 28633 genes
    Solo en raw: 70551 genes


## 5. Cadena de join RNA → CSV central

```
Sample_N (col matriz) → N = char_rnaseq_sampleid (GEO)
                      → char_study_sample_id (GEO) = Sample (CSV central)
```

In [7]:
geo_rna = pd.read_csv(CONFIG['geo_rna_meta'])
csv = pd.read_csv(CONFIG['csv_central'], low_memory=False)
csv_sample_ids = set(csv['Sample'].dropna().astype(int))

# Construir lookup: rnaseq_id → sample_id (del CSV central)
rna_lookup = dict(zip(
    geo_rna['char_rnaseq_sampleid'].astype(int),
    geo_rna['char_study_sample_id'].astype(int)
))
print(f"  Lookup rnaseq_id → sample_id: {len(rna_lookup)} entradas")
print(f"  Primeras 5: { {k: v for k, v in list(rna_lookup.items())[:5]} }")

# Verificar: ¿todos los Sample_N de la matriz tienen lookup?
matrix_rna_ids = set()
for col in rna.columns:
    col_clean = col.strip().strip('"')
    if col_clean.startswith('Sample_'):
        try:
            matrix_rna_ids.add(int(col_clean.replace('Sample_', '')))
        except:
            pass

in_lookup = matrix_rna_ids & set(rna_lookup.keys())
not_in_lookup = matrix_rna_ids - set(rna_lookup.keys())
print(f"\n  Columnas Sample_N en matriz: {len(matrix_rna_ids)}")
print(f"  Con lookup a CSV central: {len(in_lookup)}")
print(f"  Sin lookup: {len(not_in_lookup)}")
if not_in_lookup:
    print(f"  ⚠️ IDs sin lookup: {sorted(not_in_lookup)[:20]}")

# Verificar que sample_ids resultantes existen en CSV central
mapped_sample_ids = set(rna_lookup[k] for k in in_lookup)
in_csv = mapped_sample_ids & csv_sample_ids
not_in_csv = mapped_sample_ids - csv_sample_ids
print(f"\n  Sample_ids mapeados que existen en CSV: {len(in_csv)} / {len(mapped_sample_ids)}")
if not_in_csv:
    print(f"  ⚠️ NO en CSV: {sorted(not_in_csv)[:20]}")
else:
    print(f"  ✅ Todos los sample_ids mapeados existen en CSV central")

  Lookup rnaseq_id → sample_id: 345 entradas
  Primeras 5: {1: 430, 2: 433, 3: 436, 4: 439, 5: 442}

  Columnas Sample_N en matriz: 345
  Con lookup a CSV central: 345
  Sin lookup: 0

  Sample_ids mapeados que existen en CSV: 345 / 345
  ✅ Todos los sample_ids mapeados existen en CSV central


## 6. Inspección metilación (sin cargar toda la matriz)

La matriz es ~2 GB. Cargamos solo 10 filas para entender la estructura.

In [8]:
met_preview = pd.read_csv(
    CONFIG['met_processed'], compression='gzip', index_col=0, nrows=10
)
print(f"  Preview shape: {met_preview.shape}")
print(f"  Index (CpGs): {met_preview.index[:5].tolist()}")
print(f"  Index comienza con 'cg': {str(met_preview.index[0]).startswith('cg')}")

# Columnas
all_met_cols = met_preview.columns.tolist()
print(f"\n  Total columnas: {len(all_met_cols)}")
print(f"  Columna 1: {all_met_cols[0]}")
print(f"  Columna 2: {all_met_cols[1]}")
print(f"  Columna 3: {all_met_cols[2]}")

# Separar betas de p-values
beta_columns = [c for c in all_met_cols if 'beta' in c.lower()]
pval_columns = [c for c in all_met_cols if 'detection' in c.lower() or 'pval' in c.lower()]
other_columns = [c for c in all_met_cols if c not in beta_columns and c not in pval_columns]

print(f"\n  Columnas beta: {len(beta_columns)}")
print(f"  Columnas Detection Pval: {len(pval_columns)}")
print(f"  Otras: {len(other_columns)}")
if other_columns:
    print(f"    → {other_columns[:5]}")

# Extraer sentrix basenames
sentrix_basenames = []
for c in beta_columns:
    basename = c.rsplit(' ', 1)[0].strip().strip('"')
    sentrix_basenames.append(basename)

print(f"\n  Muestras (sentrix basenames) en beta cols: {len(sentrix_basenames)}")
print(f"  Ejemplos: {sentrix_basenames[:5]}")

# Valores
beta_preview = met_preview[beta_columns]
print(f"\n  Rango betas (preview): [{beta_preview.values.min():.6f}, {beta_preview.values.max():.6f}]")
print(f"  NaN en betas (preview): {beta_preview.isna().sum().sum()}")

if beta_preview.values.max() <= 1.0 and beta_preview.values.min() >= 0.0:
    print("  ✅ Valores en rango beta [0, 1]")
else:
    print("  ⚠️ Valores fuera de rango beta — verificar")

  Preview shape: (10, 958)
  Index (CpGs): ['cg14817997', 'cg26928153', 'cg16269199', 'cg13869341', 'cg14008030']
  Index comienza con 'cg': True

  Total columnas: 958
  Columna 1: 203784950092_R01C01 beta
  Columna 2: 203784950092_R01C01 Detection Pval
  Columna 3: 203784950092_R02C01 beta

  Columnas beta: 479
  Columnas Detection Pval: 479
  Otras: 0

  Muestras (sentrix basenames) en beta cols: 479
  Ejemplos: ['203784950092_R01C01', '203784950092_R02C01', '203784950092_R03C01', '203784950092_R04C01', '203784950092_R05C01']

  Rango betas (preview): [0.056925, 0.966710]
  NaN en betas (preview): 0
  ✅ Valores en rango beta [0, 1]


## 7. Cadena de join metilación → CSV central

```
"203784950092_R01C01 beta" (col matriz) → parsear sentrix_basename
  → match con char_slide + "_" + char_array (GEO)
  → char_study_sample_id (GEO) = Sample (CSV central)
```

In [9]:
geo_met = pd.read_csv(CONFIG['geo_met_meta'])

# Construir sentrix_basename en el GEO
geo_met['sentrix_basename'] = (
    geo_met['char_slide'].astype(str).str.strip()
    + '_'
    + geo_met['char_array'].astype(str).str.strip()
)

# Lookup: sentrix_basename → sample_id (del CSV central)
met_lookup = dict(zip(
    geo_met['sentrix_basename'],
    geo_met['char_study_sample_id'].astype(int)
))
print(f"  Lookup sentrix → sample_id: {len(met_lookup)} entradas")
print(f"  Primeras 3: { {k: v for k, v in list(met_lookup.items())[:3]} }")

# Verificar contra las columnas beta de la matriz
sentrix_set_matrix = set(sentrix_basenames)
sentrix_set_geo = set(met_lookup.keys())

in_lookup_met = sentrix_set_matrix & sentrix_set_geo
not_in_lookup_met = sentrix_set_matrix - sentrix_set_geo

print(f"\n  Sentrix en matriz: {len(sentrix_set_matrix)}")
print(f"  Con lookup a CSV: {len(in_lookup_met)}")
print(f"  Sin lookup: {len(not_in_lookup_met)}")
if not_in_lookup_met:
    print(f"  ⚠️ Sin lookup: {sorted(not_in_lookup_met)[:10]}")
else:
    print(f"  ✅ Todos los sentrix tienen lookup")

# Verificar sample_ids en CSV central
mapped_met_ids = set(met_lookup[k] for k in in_lookup_met)
in_csv_met = mapped_met_ids & csv_sample_ids
not_in_csv_met = mapped_met_ids - csv_sample_ids
print(f"\n  Sample_ids mapeados en CSV: {len(in_csv_met)} / {len(mapped_met_ids)}")
if not_in_csv_met:
    print(f"  ⚠️ NO en CSV: {sorted(not_in_csv_met)[:20]}")
else:
    print(f"  ✅ Todos existen en CSV central")

  Lookup sentrix → sample_id: 479 entradas
  Primeras 3: {'203784950011_R01C01': 2, '203784950064_R03C01': 4, '203784950028_R06C01': 7}

  Sentrix en matriz: 479
  Con lookup a CSV: 479
  Sin lookup: 0
  ✅ Todos los sentrix tienen lookup

  Sample_ids mapeados en CSV: 479 / 479
  ✅ Todos existen en CSV central


## 8. Pairing multimodal — visión global

¿Cuántas muestras tienen cada combinación de modalidades?

In [10]:
# Cargar manifest MVP-1 para sample_ids con imagen
manifest = pd.read_csv(CONFIG['manifest_mvp1'], low_memory=False)
img_sample_ids = set(manifest['sample_id'].dropna().astype(int).unique())

# Muestras con RNA (vía lookup)
rna_sample_ids = set(rna_lookup[k] for k in in_lookup)

# Muestras con metilación (vía lookup)
met_sample_ids = set(met_lookup[k] for k in in_lookup_met)

# Biomarcadores del CSV central
telo_sample_ids = set(csv.loc[csv['Telomere_Length'].notna(), 'Sample'].astype(int))
mtdna_sample_ids = set(csv.loc[csv['Copy_Number'].notna(), 'Sample'].astype(int))
pdl_sample_ids = set(csv.loc[csv['Population_Doublings_DI'].notna(), 'Sample'].astype(int))

print(f"  Muestras totales en CSV central: {len(csv_sample_ids)}")
print(f"  Con PDL: {len(pdl_sample_ids)}")
print()
print(f"  Con imagen:     {len(img_sample_ids)}")
print(f"  Con RNA:        {len(rna_sample_ids)}")
print(f"  Con metilación: {len(met_sample_ids)}")
print(f"  Con telómero:   {len(telo_sample_ids)}")
print(f"  Con mtDNA CN:   {len(mtdna_sample_ids)}")
print()

combos = {
    'RNA ∩ Met':                rna_sample_ids & met_sample_ids,
    'RNA ∩ Img':                rna_sample_ids & img_sample_ids,
    'Met ∩ Img':                met_sample_ids & img_sample_ids,
    'RNA ∩ Met ∩ Img':          rna_sample_ids & met_sample_ids & img_sample_ids,
    'Cualquier ómica':          rna_sample_ids | met_sample_ids,
    'Ómica sin imagen':         (rna_sample_ids | met_sample_ids) - img_sample_ids,
    'Ómica + PDL':              (rna_sample_ids | met_sample_ids) & pdl_sample_ids,
    'RNA + PDL':                rna_sample_ids & pdl_sample_ids,
    'Met + PDL':                met_sample_ids & pdl_sample_ids,
    'Telo ∩ mtDNA':             telo_sample_ids & mtdna_sample_ids,
    'RNA ∩ Met ∩ Telo':         rna_sample_ids & met_sample_ids & telo_sample_ids,
    'Todo (RNA+Met+Img+Telo+mtDNA)': rna_sample_ids & met_sample_ids & img_sample_ids & telo_sample_ids & mtdna_sample_ids,
}

for name, s in combos.items():
    print(f"  {name}: {len(s)}")

  Muestras totales en CSV central: 1919
  Con PDL: 1918

  Con imagen:     973
  Con RNA:        345
  Con metilación: 479
  Con telómero:   496
  Con mtDNA CN:   493

  RNA ∩ Met: 109
  RNA ∩ Img: 148
  Met ∩ Img: 254
  RNA ∩ Met ∩ Img: 32
  Cualquier ómica: 715
  Ómica sin imagen: 345
  Ómica + PDL: 715
  RNA + PDL: 345
  Met + PDL: 479
  Telo ∩ mtDNA: 478
  RNA ∩ Met ∩ Telo: 109
  Todo (RNA+Met+Img+Telo+mtDNA): 32


## 9. Verificar genes hallmark en la matriz RNA

¿Cuáles de los ~80 genes marcadores de hallmarks están en la matriz?

In [11]:
available_genes = set(rna.index)
print(f"  Genes totales en matriz: {len(available_genes)}")

HALLMARK_GENES = {
    'senescence': ['CDKN2A', 'CDKN1A', 'TP53', 'RB1', 'GLB1', 'LMNB1', 'HMGA1', 'HMGA2'],
    'sasp': ['IL6', 'IL8', 'CXCL8', 'IL1A', 'IL1B', 'MMP1', 'MMP3', 'MMP9',
             'SERPINE1', 'CCL2', 'IGFBP3', 'IGFBP5', 'IGFBP7'],
    'mitochondrial': ['MT-ND1', 'MT-ND2', 'MT-CO1', 'MT-CO2', 'MT-CO3',
                      'MT-ATP6', 'MT-CYB', 'TFAM', 'PPARGC1A', 'SOD2', 'PINK1'],
    'telomere': ['TERT', 'TERC', 'DKC1', 'POT1'],
    'genomic_instability': ['ATM', 'ATR', 'BRCA1', 'BRCA2', 'H2AFX', 'CHEK1', 'CHEK2'],
    'nutrient_sensing': ['MTOR', 'SIRT1', 'SIRT3', 'SIRT6', 'IGF1', 'IGF1R'],
}

total_found = 0
total_missing = 0
for category, genes in HALLMARK_GENES.items():
    found = [g for g in genes if g in available_genes]
    missing = [g for g in genes if g not in available_genes]
    total_found += len(found)
    total_missing += len(missing)
    status = "✅" if not missing else "⚠️"
    print(f"\n  {status} {category}: {len(found)}/{len(genes)}")
    if missing:
        for m in missing:
            variants = [g for g in available_genes if m.lower() in g.lower()]
            if variants:
                print(f"    ❌ {m} — posibles variantes: {variants[:5]}")
            else:
                print(f"    ❌ {m} — no encontrado")

print(f"\n  TOTAL: {total_found} encontrados, {total_missing} faltantes")

  Genes totales en matriz: 28633

  ✅ senescence: 8/8

  ⚠️ sasp: 12/13
    ❌ IL8 — no encontrado

  ⚠️ mitochondrial: 4/11
    ❌ MT-ND1 — no encontrado
    ❌ MT-ND2 — no encontrado
    ❌ MT-CO1 — no encontrado
    ❌ MT-CO2 — no encontrado
    ❌ MT-CO3 — no encontrado
    ❌ MT-ATP6 — no encontrado
    ❌ MT-CYB — no encontrado

  ✅ telomere: 4/4

  ⚠️ genomic_instability: 6/7
    ❌ H2AFX — no encontrado

  ✅ nutrient_sensing: 6/6

  TOTAL: 40 encontrados, 9 faltantes


## 10. Plan de carga de metilación

Estimar RAM necesaria y definir estrategia de carga.

In [12]:
print(f"  Columnas totales en matriz: {len(all_met_cols)}")
print(f"  Solo betas: {len(beta_columns)} (= {len(beta_columns)} muestras)")
print(f"  Solo pvals: {len(pval_columns)}")
print()

# Estimar tamaño en memoria (solo betas)
n_cpgs_est = 866091  # EPIC array
n_samples_met = len(beta_columns)
mem_float64_gb = (n_cpgs_est * n_samples_met * 8) / (1024**3)
mem_float32_gb = mem_float64_gb / 2

print(f"  Estimación RAM (solo betas):")
print(f"    float64: {mem_float64_gb:.1f} GB")
print(f"    float32: {mem_float32_gb:.1f} GB")
print()

if mem_float32_gb < 8:
    print("  📋 ESTRATEGIA: Cargar solo columnas beta en float32")
    print(f"     → pd.read_csv(..., usecols=['index'] + beta_columns, dtype=np.float32)")
elif mem_float32_gb < 16:
    print("  📋 ESTRATEGIA: Carga posible con float32 si tienes 16+ GB RAM")
else:
    print("  📋 ESTRATEGIA: Cargar por chunks o pre-seleccionar CpGs")

print(f"\n  Para el encoder necesitamos ~10K CpGs (de 866K).")
print(f"  Opción 1: Cargar todo en float32 → seleccionar por varianza")
print(f"  Opción 2: Calcular varianza por chunks → recargar solo top 10K")

  Columnas totales en matriz: 958
  Solo betas: 479 (= 479 muestras)
  Solo pvals: 479

  Estimación RAM (solo betas):
    float64: 3.1 GB
    float32: 1.5 GB

  📋 ESTRATEGIA: Cargar solo columnas beta en float32
     → pd.read_csv(..., usecols=['index'] + beta_columns, dtype=np.float32)

  Para el encoder necesitamos ~10K CpGs (de 866K).
  Opción 1: Cargar todo en float32 → seleccionar por varianza
  Opción 2: Calcular varianza por chunks → recargar solo top 10K


## 11. Diagnóstico: bug en sección 7 del manifest MVP-1

In [13]:
print("""
  EL BUG:
    Sección 7 usaba char_study_sample_id como join key para ambos,
    pero los IDs no coinciden:
    
    RNA:  rnaseq_id (rango 1-252) ≠ char_study_sample_id (rango 318-1907)
    MET:  methylation_basename (texto) ≠ char_study_sample_id (entero)
    
    → 0 matches → columnas rna_char_*/meth_char_* vacías.

  CORRECCIÓN (en seccion7_fix.py):
    RNA:  unir rnaseq_id ↔ char_rnaseq_sampleid  (ambos rango 1-360)
    MET:  construir sentrix_basename = char_slide + "_" + char_array,
          luego unir methylation_basename ↔ sentrix_basename

  IMPACTO MVP-1: Ninguno. Flags has_rna/has_methylation son correctos.
  IMPACTO MVP-2: Necesitamos lookups correctos (ya verificados arriba).
""")

# Verificar que los flags son correctos a pesar del bug
n_has_rna = manifest['has_rna'].sum()
n_rnaseq_id = manifest['rnaseq_id'].notna().sum()
rna_geo_col = 'rna_geo_accession' if 'rna_geo_accession' in manifest.columns else None
n_rna_geo = manifest[rna_geo_col].notna().sum() if rna_geo_col else 0

print(f"  has_rna=True: {n_has_rna}")
print(f"  rnaseq_id no-null: {n_rnaseq_id}")
print(f"  rna_geo_accession no-null: {n_rna_geo}  ← vacío por el bug")
print()

n_has_met = manifest['has_methylation'].sum()
n_met_bn = manifest['methylation_basename'].notna().sum()
meth_geo_col = 'meth_geo_accession' if 'meth_geo_accession' in manifest.columns else None
n_met_geo = manifest[meth_geo_col].notna().sum() if meth_geo_col else 0

print(f"  has_methylation=True: {n_has_met}")
print(f"  methylation_basename no-null: {n_met_bn}")
print(f"  meth_geo_accession no-null: {n_met_geo}  ← vacío por el bug")

print("\n  ✅ Flags correctos — no afectó resultados MVP-1")
print("  📋 Fix disponible en seccion7_fix.py")


  EL BUG:
    Sección 7 usaba char_study_sample_id como join key para ambos,
    pero los IDs no coinciden:

    RNA:  rnaseq_id (rango 1-252) ≠ char_study_sample_id (rango 318-1907)
    MET:  methylation_basename (texto) ≠ char_study_sample_id (entero)

    → 0 matches → columnas rna_char_*/meth_char_* vacías.

  CORRECCIÓN (en seccion7_fix.py):
    RNA:  unir rnaseq_id ↔ char_rnaseq_sampleid  (ambos rango 1-360)
    MET:  construir sentrix_basename = char_slide + "_" + char_array,
          luego unir methylation_basename ↔ sentrix_basename

  IMPACTO MVP-1: Ninguno. Flags has_rna/has_methylation son correctos.
  IMPACTO MVP-2: Necesitamos lookups correctos (ya verificados arriba).

  has_rna=True: 366
  rnaseq_id no-null: 366
  rna_geo_accession no-null: 366  ← vacío por el bug

  has_methylation=True: 564
  methylation_basename no-null: 564
  meth_geo_accession no-null: 558  ← vacío por el bug

  ✅ Flags correctos — no afectó resultados MVP-1
  📋 Fix disponible en seccion7_fix.py


## 12. Resumen de exploración

In [14]:
print("=" * 70)
print("RESUMEN DE EXPLORACIÓN")
print("=" * 70)
print(f"""
  RNA-seq processed:
    Shape: {rna.shape[0]} genes × {rna.shape[1]} muestras
    Columnas: Sample_1 ... Sample_{max(matrix_rna_ids)}
    Join: Sample_N → char_rnaseq_sampleid → char_study_sample_id → Sample CSV
    
  RNA-seq raw counts:
    Shape: {rna_raw.shape[0]} genes × {rna_raw.shape[1]} muestras
    
  Metilación processed:
    Filas: ~866K CpGs
    Columnas: {len(beta_columns)} betas + {len(pval_columns)} pvals
    Formato: beta [0,1] + Detection p-values  
    Join: sentrix_basename → char_study_sample_id → Sample CSV
    ⚠️ Solo cargar columnas beta (ignorar Detection Pval)
    
  Pairing multimodal:
    RNA: {len(rna_sample_ids)} | Met: {len(met_sample_ids)}
    RNA ∩ Met: {len(rna_sample_ids & met_sample_ids)}
    RNA ∩ Img: {len(rna_sample_ids & img_sample_ids)}
    RNA ∩ Met ∩ Img: {len(rna_sample_ids & met_sample_ids & img_sample_ids)}
    
  SIGUIENTE PASO:
    Construir notebook MVP-2 con estos mapeos verificados.
""")

RESUMEN DE EXPLORACIÓN

  RNA-seq processed:
    Shape: 28633 genes × 345 muestras
    Columnas: Sample_1 ... Sample_360
    Join: Sample_N → char_rnaseq_sampleid → char_study_sample_id → Sample CSV

  RNA-seq raw counts:
    Shape: 70551 genes × 345 muestras

  Metilación processed:
    Filas: ~866K CpGs
    Columnas: 479 betas + 479 pvals
    Formato: beta [0,1] + Detection p-values  
    Join: sentrix_basename → char_study_sample_id → Sample CSV
    ⚠️ Solo cargar columnas beta (ignorar Detection Pval)

  Pairing multimodal:
    RNA: 345 | Met: 479
    RNA ∩ Met: 109
    RNA ∩ Img: 148
    RNA ∩ Met ∩ Img: 32

  SIGUIENTE PASO:
    Construir notebook MVP-2 con estos mapeos verificados.

